# 03_build_features Build the training table

**Owner:** Person 1

**Goal:** Clean Ookla, add tower features, produce the file the models train on.

**Outputs:** data/interim/ookla_clean.csv, data/processed/hanoi_train.csv

In [ ]:
import sys
from pathlib import Path

# Cho phép import package speedmap khi chạy notebook từ thư mục notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from speedmap import config

## 1. Clean Ookla

In [ ]:
from speedmap.data import ookla, towers
from speedmap.features.build import build_training_table, validate_training_table

raw = ookla.load_raw()
clean = ookla.clean(raw)
print(f"{len(raw)} raw rows -> {len(clean)} clean rows ({len(raw) - len(clean)} dropped)")
config.OOKLA_CLEAN_FILE.parent.mkdir(parents=True, exist_ok=True)
clean.to_csv(config.OOKLA_CLEAN_FILE, index=False)

## 2. Add tower features

In [ ]:
train = build_training_table(clean, towers.load_towers())
validate_training_table(train)
print(train.shape)
train.head()

## 3. Sanity checks
Distance to nearest tower vs download: is there a visible trend?
`theory_mbps` stays empty (NaN) until the physics module (T-06, Person 2) is finished; then re-run this notebook.

In [ ]:
display(train[config.FEATURE_COLUMNS].describe().round(1))
print(train.quality_level.value_counts(normalize=True).round(3))

usable = train[train.tests >= config.MIN_TESTS_FOR_EVAL]
for col in ["dist_nearest_tower_m", "towers_500m", "towers_1km", "quarter_index"]:
    print(f"Spearman({col}, download) = {usable[[col, 'download_mbps']].corr('spearman').iloc[0, 1]:.3f}")

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(usable.dist_nearest_tower_m.clip(lower=10), usable.download_mbps, s=3, alpha=0.3)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("distance to nearest tower (m)"); ax.set_ylabel("download (Mbps)")
ax.set_title("Download vs distance to nearest tower (tiles with >= 3 tests)")
plt.savefig(config.ARTIFACTS_REPORTS / "features_distance_vs_download.png", dpi=150); plt.show()

## 4. Save

In [ ]:
config.TRAIN_FILE.parent.mkdir(parents=True, exist_ok=True)
train.to_csv(config.TRAIN_FILE, index=False)
print("saved", config.TRAIN_FILE)